# Memory

<div style="display: flex; justify-content: flex-start; gap: 10px;">
  <img src="https://github.com/jayyanar/agentic-ai-training/blob/lab-day-1/batch2/lca-langchainV1-essentials/assets/LC_Memory_before.png?raw=1" style="width:300px; border:1px solid #ccc; border-radius:6px;">
</div>

Persisting messages, or 'agent state' between invocations of the agent.

What we're doing: Install required packages for the memory lesson in Colab.

In [1]:
!pip install -qU langchain-groq langgraph langchain-community pysqlite3-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.0/503.0 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


What we're doing: Load the GROQ API key from Colab userdata into the environment.

In [2]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

What we're doing: Connect to the sample SQLite database and print available tables.

In [18]:
from langchain_community.utilities import SQLDatabase

# Download the database file locally
!wget -q https://github.com/jayyanar/agentic-ai-training/raw/lab-day-1/batch2/lca-langchainV1-essentials/Chinook.db

# Update the URI to the local path
db = SQLDatabase.from_uri("sqlite:///Chinook.db")

# Verify it works
print(db.get_usable_table_names())

['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


What we're doing: Define `RuntimeContext` dataclass to hold the database reference for runtime use.

In [19]:
from dataclasses import dataclass

@dataclass
class RuntimeContext:
    db: SQLDatabase

What we're doing: Define `execute_sql` tool to allow the agent to run read-only SQL queries via the runtime.

In [20]:
from langchain_core.tools import tool
from langgraph.runtime import get_runtime

@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

What we're doing: Set `SYSTEM_PROMPT` rules to constrain agent SQL behavior and ensure read-only queries.

In [21]:
SYSTEM_PROMPT = """You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
"""

What we're doing: Initialize the Groq model and create an agent with `execute_sql` and runtime context.

In [22]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq

# Initialize the Groq model
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_retries=2,
)

agent = create_agent(
    #model="openai:gpt-5",
    model=llm,
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
)

## Repeated Queries

What we're doing: Stream a repeated query scenario to observe agent behavior without memory enabled.

In [23]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
    context=RuntimeContext(db=db),
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (fc_3db3c863-cdec-4a87-ba6d-3b8b096d3ce3)
 Call ID: fc_3db3c863-cdec-4a87-ba6d-3b8b096d3ce3
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;
================================= Tool Message =================================
Name: execute_sql

[('Album',), ('Artist',), ('Customer',), ('Employee',), ('Genre',), ('Invoice',), ('InvoiceLine',), ('MediaType',), ('Playlist',), ('PlaylistTrack',), ('Track',)]
================================== Ai Message ==================================
Tool Calls:
  execute_sql (fc_3bee0a62-75bb-46b8-817d-41f5037953f8)
 Call ID: fc_3bee0a62-75bb-46b8-817d-41f5037953f8
  Args:
    query: PRAGMA table_info('Customer');
================================= Tool Message ===============

What we're doing: Follow up query without memory to show that prior context isn't retained across invocations.

In [24]:
question = "What were the titles?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (fc_3abe52e9-b89e-475c-9a84-f97ec9b6d61a)
 Call ID: fc_3abe52e9-b89e-475c-9a84-f97ec9b6d61a
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table' LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

[('Album',), ('Artist',), ('Customer',), ('Employee',), ('Genre',)]
================================== Ai Message ==================================
Tool Calls:
  execute_sql (fc_78165680-7aa9-4c6e-9199-4e19d4a97ad0)
 Call ID: fc_78165680-7aa9-4c6e-9199-4e19d4a97ad0
  Args:
    query: PRAGMA table_info('Album');
================================= Tool Message =================================
Name: execute_sql

[(0, 'AlbumId', 'INTEGER', 1, None, 1), (1, 'Title', 'NVARCHAR(160)', 1, None, 0), (2, 'ArtistId', 'INTEG

## Add memory

What we're doing: Import an in-memory checkpointer to enable simple agent memory persistence.

In [25]:
from langgraph.checkpoint.memory import InMemorySaver

What we're doing: Re-create the agent with an in-memory checkpointer so it can remember prior interactions.

In [26]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage

agent = create_agent(
    #model="openai:gpt-5",
    model=llm,
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
    checkpointer=InMemorySaver(),
)

What we're doing: Run the same invoice query with memory enabled (thread_id=1) to persist context between calls.

In [27]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (fc_7412cec6-f2f1-42a8-83d2-36e2b64b1df9)
 Call ID: fc_7412cec6-f2f1-42a8-83d2-36e2b64b1df9
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;
================================= Tool Message =================================
Name: execute_sql

[('Album',), ('Artist',), ('Customer',), ('Employee',), ('Genre',), ('Invoice',), ('InvoiceLine',), ('MediaType',), ('Playlist',), ('PlaylistTrack',), ('Track',)]
================================== Ai Message ==================================
Tool Calls:
  execute_sql (fc_617532f6-3d0d-4059-a63f-7b40800b4e97)
 Call ID: fc_617532f6-3d0d-4059-a63f-7b40800b4e97
  Args:
    query: PRAGMA table_info('Customer');
================================= Tool Message ===============

What we're doing: Ask a follow-up question using the same memory `thread_id` to check recall of details like titles.

In [28]:
question = "What were the titles?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (fc_4bacf143-aeb5-4e36-bc05-3fa018ab5bb7)
 Call ID: fc_4bacf143-aeb5-4e36-bc05-3fa018ab5bb7
  Args:
    query: SELECT InvoiceId FROM Invoice WHERE CustomerId = 16 ORDER BY InvoiceDate DESC LIMIT 1;
================================= Tool Message =================================
Name: execute_sql

[(374,)]
================================== Ai Message ==================================
Tool Calls:
  execute_sql (fc_0b71b992-21f9-45a7-aac0-addea856a416)
 Call ID: fc_0b71b992-21f9-45a7-aac0-addea856a416
  Args:
    query: SELECT t.Name FROM InvoiceLine il JOIN Track t ON il.TrackId = t.TrackId WHERE il.InvoiceId = 374 LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

[('Holier Than Thou',), ('Through The Never',), ('My Frie

## Try your own queries
Now that there is memory, check the agents recall!

What we're doing: Placeholder for you to enter your own question and test agent memory behavior.

In [30]:
question = "This is Dan Miller. Give me the list of my invoices?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Dan Miller. Give me the list of my invoices?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (fc_52e97c40-99b8-4e74-ae25-4258a4afe6b3)
 Call ID: fc_52e97c40-99b8-4e74-ae25-4258a4afe6b3
  Args:
    query: SELECT CustomerId FROM Customer WHERE FirstName = 'Dan' AND LastName = 'Miller';
================================= Tool Message =================================
Name: execute_sql

[(20,)]
================================== Ai Message ==================================
Tool Calls:
  execute_sql (fc_908a784c-2d15-44d8-8a12-c2b77dbc4e65)
 Call ID: fc_908a784c-2d15-44d8-8a12-c2b77dbc4e65
  Args:
    query: SELECT InvoiceId, InvoiceDate, Total FROM Invoice WHERE CustomerId = 20 ORDER BY InvoiceDate;
================================= Tool Message =================================
Name: execute_sql

[(113, '2010-05-12 00:00:00', 1.98), (124, '20

What we're doing: Example follow-up that asks the agent to show full results using the same memory thread.

In [31]:
question = "I want to see the title name of my most expensive invoice"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

I want to see the title name of my most expensive invoice
================================== Ai Message ==================================
Tool Calls:
  execute_sql (fc_4bfef89d-3b6f-4683-9c65-7c800479ff29)
 Call ID: fc_4bfef89d-3b6f-4683-9c65-7c800479ff29
  Args:
    query: SELECT t.Name FROM InvoiceLine il JOIN Track t ON il.TrackId = t.TrackId WHERE il.InvoiceId = 124;
================================= Tool Message =================================
Name: execute_sql

[('Posso Perder Minha Mulher, Minha Mãe, Desde Que Eu Tenha O Rock And Roll',), ('You Fool No One',), ('Viradouro',), ('Tradição',), ('Maria Maria',), ('Entre E Ouça',), ('Agora Que O Dia Acordou',), ('Bye Bye Blackbird',), ('Nefertiti',), ('Going Down / Highway Star',), ('Boogie Blues',), ("Don't Take Your Love From Me",), ('Onde Anda Você',), ('Deixa',)]
================================== Ai Message ==================================

**

(End of notebook)